# Chapter 3 — Astrometric orbits: validation, masses, and Gaia BH1

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/gaia-nss-tutorial/blob/main/notebooks/03_astrometric_orbits.ipynb)

**Goal.** Gaia DR3 contains 134 598 astrometric orbits (`Orbital`) and 33 467 combined astrometric–spectroscopic ones
(`AstroSpectroSB1`). Each gives the photocentre orbit as Thiele–Innes constants. In this chapter we turn them into
orbital elements with uncertainties, check how they were filtered, convert them into a mass function and a
companion mass, and sort the companions into those that could be ordinary stars and those that cannot (the
"astrometric mass-ratio function" triage of Shahaf et al. 2019, https://ui.adsabs.harvard.edu/abs/2019MNRAS.487.5610S,
and 2023, https://ui.adsabs.harvard.edu/abs/2023MNRAS.518.2991S). We finish with Gaia BH1, the first dormant black hole
found by Gaia (El-Badry et al. 2023, https://ui.adsabs.harvard.edu/abs/2023MNRAS.518.1057E).

**You will learn**
1. to convert Thiele–Innes constants to Campbell elements, and to propagate the published correlation matrix;
2. which filters an orbit had to pass to be published, and what they imply for the catalogue (doubt first);
3. to compute the photocentre mass function, and why a dark companion and a luminous one give different answers;
4. what the AMRF classes mean — and what they do not;
5. that a Gaia-only mass and a mass from Gaia plus spectroscopy can differ, and why.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.optimize import brentq                # root of the mass-function equation

# --- Data access (identical in every chapter) -------------------------------------------------------------
DATA_URL = 'https://github.com/vasilybelokurov/gaia-nss-tutorial/releases/download/data-v0.2/'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def load(name, cache=Path('data')):
    """Download <name>.parquet from the tutorial's data release (again if the local copy is stale), check its SHA-256."""
    path = cache / f'{name}.parquet'
    if not path.exists():                          # first use: fetch the file into ./data
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != MANIFEST[name]['sha256']:   # stale or damaged copy
        print(f'downloading {path.name} again (checksum mismatch) ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    # the checksum guarantees that everyone runs this chapter on exactly the same rows
    assert hashlib.sha256(path.read_bytes()).hexdigest() == MANIFEST[name]['sha256'], f'{path} does not match the manifest'
    return pd.read_parquet(path)


tbo = load('nss_two_body')
orb = tbo[tbo.nss_solution_type == 'Orbital'].reset_index(drop=True)
BH1, BH2 = 4373465352415301632, 5870569352746779008
print(f'{len(orb):,} Orbital solutions')

## 3.1 From Thiele–Innes constants to Campbell elements

Gaia fits the photocentre orbit with the Thiele–Innes constants $A, B, F, G$ (Chapter 1), because the model is linear
in them. The inverse relations (Halbwachs et al. 2023, Appendix A, https://ui.adsabs.harvard.edu/abs/2023A&A...674A...9H):
with $u = (A^2 + B^2 + F^2 + G^2)/2$ and $v = AG - BF$,

$$ a_0 = \sqrt{u + \sqrt{(u+v)(u-v)}}, \qquad \cos i = v / a_0^2, $$
$$ \omega + \Omega = \operatorname{atan2}(B - F, A + G), \qquad \omega - \Omega = \operatorname{atan2}(-B - F, A - G), $$

with the convention $0 \le \Omega < 180°$ (astrometry cannot distinguish $(\omega, \Omega)$ from
$(\omega + 180°, \Omega + 180°)$). We test the pair of transformations on random orbits before using them. The
comparison is made on the Thiele–Innes constants, not on the angles, which wrap around and are degenerate for
face-on or circular orbits.

In [ ]:
def thiele_innes(a0, i, omega, Omega):
    """Thiele-Innes constants A, B, F, G (units of a0) from Campbell elements (angles in degrees)."""
    i, omega, Omega = np.radians(i), np.radians(omega), np.radians(Omega)
    A = a0 * (np.cos(omega) * np.cos(Omega) - np.sin(omega) * np.sin(Omega) * np.cos(i))
    B = a0 * (np.cos(omega) * np.sin(Omega) + np.sin(omega) * np.cos(Omega) * np.cos(i))
    F = -a0 * (np.sin(omega) * np.cos(Omega) + np.cos(omega) * np.sin(Omega) * np.cos(i))
    G = -a0 * (np.sin(omega) * np.sin(Omega) - np.cos(omega) * np.cos(Omega) * np.cos(i))
    return A, B, F, G


def campbell(A, B, F, G):
    """Campbell elements (a0, i [deg], omega [deg], Omega [deg]) from Thiele-Innes constants, with 0 <= Omega < 180."""
    u, v = (A**2 + B**2 + F**2 + G**2) / 2, A * G - B * F
    a0 = np.sqrt(u + np.sqrt((u + v) * (u - v)))
    i = np.degrees(np.arccos(v / a0**2))
    s, d = np.arctan2(B - F, A + G), np.arctan2(-B - F, A - G)              # omega + Omega, omega - Omega
    om, Om = np.degrees((s + d) / 2), np.degrees((s - d) / 2)
    flip = (Om % 360) >= 180
    return a0, i, (om + 180 * flip) % 360, (Om + 180 * flip) % 360


rng = np.random.default_rng(1)
n = 100000
test = (rng.uniform(0.1, 10, n), np.degrees(np.arccos(rng.uniform(-1, 1, n))), rng.uniform(0, 360, n), rng.uniform(0, 360, n))
TI = np.array(thiele_innes(*test))
back = np.array(thiele_innes(*campbell(*TI)))
print(f'max |Thiele-Innes after the round trip - before| = {np.abs(back - TI).max():.2e} mas')
# identity: TI -> Campbell -> TI is exact to rounding error
assert np.abs(back - TI).max() < 1e-9

## 3.2 Uncertainties from the correlation matrix

The errors of $A, B, F, G$ are strongly correlated, so $\sigma(a_0)$ cannot be computed from the four errors alone.
Each solution carries its correlation matrix in `corr_vec`: the upper triangle, stored column by column, of the
correlation matrix of the fitted parameters, which for an eccentric `Orbital` solution (`bit_index` = 8191) are, in
order, $(\alpha, \delta, \varpi, \mu_{\alpha*}, \mu_\delta, A, B, F, G, e, P, T_p)$ (Gaia DR3 documentation,
https://gea.esac.esa.int/archive/documentation/GDR3/Gaia_archive/chap_datamodel/sec_dm_non--single_stars_tables/ssec_dm_nss_two_body_orbit.html).
We check the storage order on the data: a correlation matrix must be positive definite, and a wrong unpacking order
generally breaks that.

In [ ]:
PARS = ['ra', 'dec', 'parallax', 'pmra', 'pmdec', 'a_thiele_innes', 'b_thiele_innes', 'f_thiele_innes',
        'g_thiele_innes', 'eccentricity', 'period', 't_periastron']
N_P = len(PARS)
COL_MAJOR = [(i, j) for j in range(N_P) for i in range(j)]          # (0,1), (0,2), (1,2), (0,3), ...
ROW_MAJOR = [(i, j) for i in range(N_P) for j in range(i + 1, N_P)]  # (0,1), (0,2), ..., (0,11), (1,2), ...


def corr_matrix(cv, order):
    """12 x 12 correlation matrix from a corr_vec (66 values) unpacked in the given (i, j) order."""
    C = np.eye(N_P)
    for (i, j), c in zip(order, cv):
        C[i, j] = C[j, i] = c
    return C


ecc = orb[orb.bit_index == 8191].reset_index(drop=True)
sample = ecc.sample(2000, random_state=0)
pd_frac = {name: np.mean([np.linalg.eigvalsh(corr_matrix(cv, o)).min() > 0 for cv in sample.corr_vec])
           for name, o in [('column-major', COL_MAJOR), ('row-major', ROW_MAJOR)]}
print('fraction of positive-definite correlation matrices:', pd_frac)
# the documented (column-major) order gives valid correlation matrices; the other order does not
assert pd_frac['column-major'] > 0.99 > pd_frac['row-major']

With the correlation matrix $\mathsf R$ and the errors $\sigma_k$, the covariance is $\mathsf C_{kl} = \sigma_k \sigma_l
\mathsf R_{kl}$. We propagate it to $a_0$ and the other Campbell elements by Monte Carlo: draw parameter vectors from a
multivariate normal with this covariance, transform each draw, and take the standard deviation. For Gaia BH1, El-Badry
et al. (2023, Table 1, "astrometry only") quote $a_0 = 3.00 \pm 0.22$ mas, $P = 185.77 \pm 0.31$ d, $e = 0.49 \pm 0.07$,
$i = 121.2 \pm 2.8°$ from this same DR3 solution.

In [ ]:
def draws(row, n=20000, seed=0):
    """n Monte Carlo draws (n, 12) of the parameters of one eccentric Orbital solution, from its full covariance."""
    # errors of position and proper motion are set to 0: they do not enter a0, and the extract does not carry them
    err = np.array([0, 0, row.nss_parallax_error, 0, 0] + [row[f'{p}_error'] for p in PARS[5:]], float)
    mean = np.array([0, 0, row.nss_parallax, 0, 0] + [row[p] for p in PARS[5:]], float)
    C = corr_matrix(row.corr_vec, COL_MAJOR) * np.outer(err, err)
    return np.random.default_rng(seed).multivariate_normal(mean, C, n, method='eigh')


bh1 = ecc.set_index('source_id').loc[BH1]
X = draws(bh1)
a0_d, i_d, om_d, Om_d = campbell(*X[:, 5:9].T)
a0, inc, om, Om = campbell(bh1.a_thiele_innes, bh1.b_thiele_innes, bh1.f_thiele_innes, bh1.g_thiele_innes)
print(f'Gaia BH1: a0 = {a0:.3f} ± {a0_d.std():.3f} mas, P = {bh1.period:.2f} ± {bh1.period_error:.2f} d, '
      f'e = {bh1.eccentricity:.3f} ± {bh1.eccentricity_error:.3f}, i = {inc:.1f} ± {i_d.std():.1f} deg')
# reproducibility: El-Badry et al. (2023), Table 1, astrometry only
assert abs(a0 - 3.00) < 0.05 and abs(a0_d.std() / 0.22 - 1) < 0.3 and abs(inc - 121.2) < 1.0

Only the parallax and the orbital parameters matter for $a_0$, so the draws above leave the errors of position and
proper motion at zero.

## 3.3 Doubt first: how the orbits were filtered

Fitting an orbit with 12 parameters to a few hundred one-dimensional measurements of a star that was preselected for
a poor single-star fit (RUWE > 1.4, Chapter 0) produces many spurious solutions. Halbwachs et al. (2023, Table 1)
therefore published an eccentric orbital solution only if

- its significance $a_0/\sigma_{a_0} > \max(5, 158/\sqrt{P/{\rm d}})$ (column `significance`);
- its goodness of fit $F_2 < 25$ (`goodness_of_fit`);
- its parallax was significant, $\varpi/\sigma_\varpi > 20\,000/(P/{\rm d})$ (this removes many short-period spurious
  orbits of distant stars);
- its eccentricity was well determined, $\sigma_e < 0.079\ln(P/{\rm d}) - 0.244$.

Figure 3.1 shows the published solutions against these limits.

In [ ]:
P = ecc.period
lim_s = np.maximum(5, 158 / np.sqrt(P))
lim_plx = 20000 / P
lim_e = 0.079 * np.log(P) - 0.244
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
pg = np.logspace(1, 3.5, 200)
for a, y, lim, lab in [(ax[0], ecc.significance, np.maximum(5, 158 / np.sqrt(pg)), '$a_0/\\sigma_{a_0}$'),
                       (ax[1], ecc.nss_parallax / ecc.nss_parallax_error, 20000 / pg, '$\\varpi/\\sigma_\\varpi$ (NSS)'),
                       (ax[2], ecc.eccentricity_error, 0.079 * np.log(pg) - 0.244, '$\\sigma_e$')]:
    a.hist2d(np.log10(P), np.log10(y), bins=150, norm=LogNorm(), cmap='viridis')
    a.plot(np.log10(pg), np.log10(np.clip(lim, 1e-3, None)), 'r-')
    a.set(xlabel='log$_{10}$ P [d]', ylabel=f'log$_{{10}}$ {lab}')
plt.tight_layout(); plt.show()
passes = (ecc.significance > lim_s) & (ecc.goodness_of_fit < 25) & (ecc.nss_parallax / ecc.nss_parallax_error > lim_plx) \
         & (ecc.eccentricity_error < lim_e)
print(f'eccentric Orbital solutions passing all four filters: {passes.mean():.4f} of {len(ecc):,}')
# descriptive: the published eccentric solutions satisfy the published filters (allowing for float32 rounding)
assert passes.mean() > 0.995

**Figure 3.1.** Eccentric `Orbital` solutions (`bit_index` = 8191) against period: significance $a_0/\sigma_{a_0}$
(left), NSS parallax S/N (middle), eccentricity error (right), log-scaled counts. Red: the publication limits of
Halbwachs et al. (2023); solutions were kept above the red line (left, middle) or below it (right).

Every one of the 132 759 eccentric solutions lies on the accepted side of all four limits, and the densest parts of the
distributions press against the limits in significance and parallax S/N (left, middle): many more solutions were
computed than published, and the published ones are those that just made it, as well as the clear detections. The
vertical gaps — near $\log P = 2.56$ (one year) and, narrower, near 2.26 — are periods where few solutions survived;
Figure 3.2 shows them directly.

The filters shape the catalogue: the hard edges in Figure 3.1 are not astrophysics. Two more features need care.

In [ ]:
logP = np.log10(orb.period)
h, e_ = np.histogram(logP, np.arange(1, 3.6, 0.02))
c = 0.5 * (e_[1:] + e_[:-1])
plt.step(10**c, h, where='mid')
plt.axvline(365.25, color='r', ls='--', lw=1, label='1 yr'); plt.axvline(1038, color='k', ls=':', lw=1, label='DR3 baseline')
plt.xscale('log'); plt.xlabel('period [d]'); plt.ylabel('Orbital solutions per 0.02 dex'); plt.legend(); plt.show()
near = np.abs(c - np.log10(365.25)) < 0.02
side = (np.abs(c - np.log10(365.25)) > 0.06) & (np.abs(c - np.log10(365.25)) < 0.16)
ratio = h[near].mean() / h[side].mean()
print(f'solutions per bin within 0.02 dex of 1 yr / in 0.06-0.16 dex on either side: {ratio:.2f}')
# descriptive: a deep deficit of orbits with periods of one year
assert ratio < 0.5

**Figure 3.2.** Period distribution of all `Orbital` solutions (0.02-dex bins). Dashed: 1 yr; dotted: the DR3 baseline.

Around one year there are 16 times fewer orbits per bin than 0.06–0.16 dex away. A binary with $P = 1$ yr moves its
photocentre in step with the parallactic ellipse, and over a few years the orbit and the parallax are nearly
degenerate: such solutions have large parallax errors and fail the parallax filter. Holl et al. (2023b,
https://ui.adsabs.harvard.edu/abs/2023A&A...674A..25H) also show that signals tied to Gaia's scanning (scan-angle
dependent image effects, the 63-d precession of the scanning law) create spurious periods; filters remove most
solutions near these periods, real or not. The cut-off near the baseline (1038 d) is the other edge: longer orbits
are rarely fitted. The period distribution of DR3 orbits is therefore the true distribution times a strongly
period-dependent selection function (Lam et al. 2025, https://ui.adsabs.harvard.edu/abs/2025ApJ...987..215L).

Second, ESA has declared four DR3 orbital solutions false: the stars are real, but the orbits are not
(https://www.cosmos.esa.int/web/gaia/dr3-known-issues).

In [ ]:
FALSE = {4698424845771339520: 'WD 0141-675', 5765846127180770432: 'HIP 64690', 522135261462534528: '54 Cas',
         1712614124767394816: 'HIP 66074'}
fs = tbo[tbo.source_id.isin(FALSE)].copy()
fs['name'] = fs.source_id.map(FALSE)
fs['a0'] = campbell(fs.a_thiele_innes, fs.b_thiele_innes, fs.f_thiele_innes, fs.g_thiele_innes)[0]
fs['f_M'] = (fs.a0 / fs.nss_parallax)**3 / (fs.period / 365.25)**2           # photocentre mass function [Msun]
print(fs[['name', 'nss_solution_type', 'phot_g_mean_mag', 'period', 'eccentricity', 'significance', 'a0', 'nss_parallax',
          'f_M']].round(3).to_string(index=False))
assert len(fs) == 4

All four are `OrbitalTargetedSearch` solutions: orbits fitted to stars from lists of candidate exoplanet hosts and
known binaries, whatever their RUWE, with relaxed filters (Holl et al. 2023a, https://ui.adsabs.harvard.edu/abs/2023A&A...674A..10H).
Their significances are low (4.7–14), their orbits small ($a_0 \approx 0.2$–0.8 mas) and their mass functions tiny,
$f_M < 0.01\,M_\odot$: they claimed substellar companions. Low-amplitude orbits found where one looks for them are
where false positives concentrate. Most of the extreme high-mass-function orbits followed up from the ground also
turned out to be wrong, apart from Gaia BH1, BH2 and the neutron-star candidate NS1 (Simon et al. 2026,
https://ui.adsabs.harvard.edu/abs/2026ApJ..1007..160S; El-Badry et al. 2026,
https://ui.adsabs.harvard.edu/abs/2026arXiv260806453E).

## 3.4 The mass function

For a photocentre orbit, Kepler's third law gives the **astrometric (photocentric) mass function**

$$ f_M = \left(\frac{a_0}{\varpi}\right)^3 \left(\frac{P}{\rm yr}\right)^{-2} M_\odot = (M_1 + M_2)\,|B - \beta|^3 , $$

with $a_0/\varpi$ in au (Chapter 1). For a dark companion ($\beta = 0$) this is $M_2^3/(M_1+M_2)^2$, which can be solved
for $M_2$ given $M_1$; a luminous companion makes $f_M$ smaller, so $M_2$ computed as if it were dark is a lower limit.
Unlike the spectroscopic mass function (Chapter 4), $f_M$ contains no unknown inclination.

To estimate $M_1$ we use main-sequence stars: we interpolate the mass as a function of $M_G$ along a PARSEC isochrone
of solar metallicity and 3.2 Gyr (`isochrones_gaia_solar`). We keep stars within 500 pc (NSS parallax > 2 mas), where
extinction is small, and within 1 mag of the isochrone at their colour.

In [ ]:
iso = load('isochrones_gaia_solar')
ms = iso[np.isclose(iso.age, 3.16e9) & (iso.label == 1)].sort_values('mini')       # main sequence (PARSEC label 1)
ms = ms[ms.gmag.diff().fillna(-1) < 0]                                              # keep M_G strictly decreasing with mass
mass_of_MG = lambda MG: np.interp(-MG, -ms.gmag.to_numpy(), ms.mini.to_numpy(), left=np.nan, right=np.nan)
colour_iso = lambda MG: np.interp(-MG, -ms.gmag.to_numpy(), (ms.g_bpmag - ms.g_rpmag).to_numpy(), left=np.nan, right=np.nan)

d = ecc[ecc.nss_parallax > 2].copy()
d['MG'] = d.phot_g_mean_mag + 5 * np.log10(d.nss_parallax) - 10
d['a0'] = campbell(d.a_thiele_innes, d.b_thiele_innes, d.f_thiele_innes, d.g_thiele_innes)[0]
on_ms = np.abs(d.bp_rp - colour_iso(d.MG)) < 0.15          # within 0.15 mag in colour, i.e. ~1 mag in M_G, of the isochrone
d = d[on_ms].copy()
d['M1'] = mass_of_MG(d.MG)
d = d[d.M1.between(0.4, 1.4)]
d['A'] = (d.a0 / d.nss_parallax) * d.M1**(-1 / 3) * (d.period / 365.25)**(-2 / 3)      # AMRF
print(f'{len(d):,} main-sequence primaries with 0.4 < M1 < 1.4 Msun within 500 pc')

## 3.5 The astrometric mass-ratio function

Dividing out the primary mass and the period leaves a quantity that depends only on the mass ratio and the flux
ratio (Shahaf et al. 2019):

$$ \mathcal A \equiv \frac{a_0}{\varpi} \left(\frac{M_1}{M_\odot}\right)^{-1/3} \left(\frac{P}{\rm yr}\right)^{-2/3}
= \frac{|q - \ell|}{(1+q)^{2/3}(1+\ell)} . $$

- A **main-sequence companion** has $\ell = L_G(qM_1)/L_G(M_1)$; $\mathcal A$ has a maximum $\mathcal A_{\rm MS}(M_1)$
  over $0 < q \le 1$.
- A companion that is itself a close pair of main-sequence stars (a **hierarchical triple**) of total mass $qM_1$ is
  fainter for its mass: two stars of mass $qM_1/2$. Its maximum is $\mathcal A_{\rm tr}(M_1) > \mathcal A_{\rm MS}$.
- A **dark companion** has $\ell = 0$ and $\mathcal A = q/(1+q)^{2/3}$, growing without limit.

Class I ($\mathcal A < \mathcal A_{\rm MS}$) is compatible with a main-sequence companion (a compact one is not
excluded); class II ($\mathcal A_{\rm MS} < \mathcal A < \mathcal A_{\rm tr}$) is incompatible with a single
main-sequence companion but compatible with a triple or a compact object; class III ($\mathcal A > \mathcal A_{\rm tr}$)
is incompatible with both, under the adopted mass–luminosity relation. Class III is therefore a selection of
*candidates* for compact companions; it is not a confirmation.

In [ ]:
L_G = lambda m: 10**(-0.4 * np.interp(m, ms.mini, ms.gmag, left=np.inf, right=np.nan))   # G-band luminosity (0 below the grid)
qq = np.linspace(0.01, 1, 400)


def amrf_limits(M1):
    """Maximum AMRF for a single main-sequence companion and for an equal-mass main-sequence pair, over 0 < q <= 1."""
    l_ms = L_G(qq * M1) / L_G(M1)
    l_tr = 2 * L_G(qq * M1 / 2) / L_G(M1)
    f = lambda l: np.abs(qq - l) / ((1 + qq)**(2 / 3) * (1 + l))
    return np.nanmax(f(l_ms)), np.nanmax(f(l_tr))


m_grid = np.linspace(0.4, 1.4, 41)
A_ms, A_tr = np.array([amrf_limits(m) for m in m_grid]).T
d['cls'] = np.where(d.A < np.interp(d.M1, m_grid, A_ms), 1, np.where(d.A < np.interp(d.M1, m_grid, A_tr), 2, 3))
print(d.cls.value_counts().sort_index().to_string())

plt.figure(figsize=(8, 6))
plt.hist2d(d.M1, d.A, bins=[80, 120], range=[[0.4, 1.4], [0, 1.2]], norm=LogNorm(), cmap='Greys')
plt.plot(m_grid, A_ms, 'C0-', lw=2, label='$\\mathcal{A}_{\\rm MS}$: main-sequence companion, maximum')
plt.plot(m_grid, A_tr, 'C1-', lw=2, label='$\\mathcal{A}_{\\rm tr}$: main-sequence pair, maximum')
for qd, ls in [(0.6, ':'), (1.0, '--')]:
    plt.axhline(qd / (1 + qd)**(2 / 3), color='C3', ls=ls, lw=1, label=f'dark companion, q = {qd}')
plt.xlabel('$M_1$ [$M_\\odot$]'); plt.ylabel('AMRF $\\mathcal{A}$'); plt.legend(fontsize=8, loc='upper left'); plt.show()

**Figure 3.3.** AMRF against primary mass for the eccentric `Orbital` solutions with main-sequence primaries within
500 pc (log-scaled counts). Blue: the maximum AMRF of a main-sequence companion; orange: of a companion that is an
equal-mass main-sequence pair; red: dark companions with $q = 0.6$ (dotted) and $q = 1$ (dashed).

Most systems are in class I (17 796 of 23 406), below the blue line, as expected for ordinary main-sequence
companions; 4098 are in class II and 1512 in class III. The striking feature is a separate, narrow band at
$\mathcal A \approx 0.5$–0.6 for $M_1 \approx 0.6$–0.9 $M_\odot$, which Shahaf et al. (2023, their Fig. 5) found too and
Shahaf et al. (2024, https://ui.adsabs.harvard.edu/abs/2024MNRAS.529.3729S) identified as white-dwarf companions:
a dark companion of about $0.6\,M_\odot$ around a K dwarf has $q \approx 0.8$ and $\mathcal A \approx 0.55$. For
$M_1 < 0.75\,M_\odot$ this band lies above $\mathcal A_{\rm tr}$, in class III. So class III is *not* a black-hole or
neutron-star sample: at these masses it is dominated by white dwarfs, and it also collects any system whose $M_1$ is
overestimated (e.g. because the companion adds light) or whose orbit is wrong. The class boundaries also depend on the
adopted isochrone; Shahaf et al. (2023) assign class probabilities instead of hard classes.

## 3.6 Gaia BH1

Gaia BH1 (`4373465352415301632`, $G = 13.8$, 480 pc) is a G dwarf of $M_1 = 0.93 \pm 0.05\,M_\odot$ (El-Badry et al.
2023). Its DR3 orbit gives:

In [ ]:
plx = X[:, 2]
fM_d = (a0_d / plx)**3 / (X[:, 10] / 365.25)**2                        # mass function of each Monte Carlo draw
M1_d = np.random.default_rng(1).normal(0.93, 0.05, len(fM_d))
ok = fM_d > 0
M2_d = np.array([brentq(lambda m: m**3 / (m1 + m)**2 - f, 1e-3, 1e4) for f, m1 in zip(fM_d[ok][:5000], M1_d[ok][:5000])])
A_bh1 = (a0 / bh1.nss_parallax) * 0.93**(-1 / 3) * (bh1.period / 365.25)**(-2 / 3)
print(f'Gaia BH1: f_M = {np.median(fM_d):.2f} Msun (16-84%: {np.percentile(fM_d, 16):.2f}-{np.percentile(fM_d, 84):.2f}); '
      f'M2 = {np.median(M2_d):.1f} +{np.percentile(M2_d, 84) - np.median(M2_d):.1f} '
      f'-{np.median(M2_d) - np.percentile(M2_d, 16):.1f} Msun; AMRF = {A_bh1:.2f} '
      f'(A_tr at 0.93 Msun: {np.interp(0.93, m_grid, A_tr):.2f})')
# reproducibility: El-Badry et al. (2023) astrometry only M2 = 12.8 +- 2.0 Msun; and BH1 is in class III
assert abs(np.median(M2_d) - 12.8) < 2.0 and A_bh1 > np.interp(0.93, m_grid, A_tr)

With $M_1 = 0.93\,M_\odot$, the DR3 orbit alone implies $f_M \approx 11.6\,M_\odot$ and a dark companion of
$M_2 \approx 13\,M_\odot$, reproducing the astrometry-only value of El-Badry et al. (2023, $12.8 \pm 2.0\,M_\odot$), with
an AMRF of 2.3, four times the class-III limit. A main-sequence star of $13\,M_\odot$ would outshine the G dwarf by
orders of magnitude, so the companion is dark whatever its exact mass.

Radial velocities then refined it. Adding ground-based RVs to the Gaia constraints, El-Badry et al. (2023) found
$a_0 = 2.67 \pm 0.02$ mas instead of $3.00 \pm 0.22$, and $M_2 = 9.62 \pm 0.18\,M_\odot$; ESPRESSO data later gave
$9.27 \pm 0.10\,M_\odot$ (Nagarajan et al. 2024, https://ui.adsabs.harvard.edu/abs/2024PASP..136a4202N). The Gaia-only
$a_0$ was 1.5σ too large, and because $M_2$ scales roughly as $a_0^3$ the mass was 33% too high. A benchmark must name
the data and the inference it comes from.

## Summary

- Thiele–Innes constants are converted to Campbell elements by closed formulae; uncertainties need the full
  correlation matrix (`corr_vec`, column-major upper triangle), propagated by Monte Carlo.
- Published orbits passed strict filters in significance, parallax S/N and eccentricity error; the catalogue's edges,
  and its deficit at one year, are features of the processing.
- The photocentre mass function needs no inclination; with a primary mass it gives $M_2$ for a dark companion, and a
  lower limit otherwise. The AMRF classes say which kinds of companion are *compatible* with an orbit.
- Gaia BH1's DR3 orbit alone implies a dark companion of $\approx 13\,M_\odot$; radial velocities lowered it to
  $9.6\,M_\odot$ while confirming that it is dark.

**Exercises.** (1) Compute the AMRF of Gaia BH2 (`5870569352746779008`, an `AstroSpectroSB1` solution) with
$M_1 = 1.07\,M_\odot$ (El-Badry et al. 2023b, https://ui.adsabs.harvard.edu/abs/2023MNRAS.521.4323E). Why is the main-sequence
AMRF triage not appropriate for this giant? (2) How do the class fractions change for an isochrone of 1 Gyr?
(3) Plot the `OrbitalTargetedSearch` solutions in Figure 3.1. Which of them would fail the main-pipeline filters?

## References

- Halbwachs et al. (2023), *Gaia Data Release 3. Astrometric binary star processing*, A&A 674, A9: https://ui.adsabs.harvard.edu/abs/2023A&A...674A...9H
- Holl et al. (2023), *Gaia Data Release 3. Astrometric orbit determination with Markov chain Monte Carlo and genetic algorithms*, A&A 674, A10: https://ui.adsabs.harvard.edu/abs/2023A&A...674A..10H
- Holl et al. (2023), *Gaia Data Release 3. Gaia scan-angle-dependent signals and spurious periods*, A&A 674, A25: https://ui.adsabs.harvard.edu/abs/2023A&A...674A..25H
- Gaia Collaboration, Arenou et al. (2023), *Gaia Data Release 3. Stellar multiplicity, a teaser for the hidden treasure*, A&A 674, A34: https://ui.adsabs.harvard.edu/abs/2023A&A...674A..34G
- Shahaf et al. (2019), *Triage of astrometric binaries - how to find triple systems and dormant black hole secondaries in the Gaia orbits*, MNRAS 487, 5610: https://ui.adsabs.harvard.edu/abs/2019MNRAS.487.5610S
- Shahaf et al. (2023), *Triage of the Gaia DR3 astrometric orbits - I. A sample of binaries with probable compact companions*, MNRAS 518, 2991: https://ui.adsabs.harvard.edu/abs/2023MNRAS.518.2991S
- Shahaf et al. (2024), *Triage of the Gaia DR3 astrometric orbits. II. A census of white dwarfs*, MNRAS 529, 3729: https://ui.adsabs.harvard.edu/abs/2024MNRAS.529.3729S
- El-Badry et al. (2023), *A Sun-like star orbiting a black hole*, MNRAS 518, 1057: https://ui.adsabs.harvard.edu/abs/2023MNRAS.518.1057E
- El-Badry et al. (2023), *A red giant orbiting a black hole*, MNRAS 521, 4323: https://ui.adsabs.harvard.edu/abs/2023MNRAS.521.4323E
- Nagarajan et al. (2024), *ESPRESSO Observations of Gaia BH1: High-precision Orbital Constraints and no Evidence for an Inner Binary*, PASP 136, 014202: https://ui.adsabs.harvard.edu/abs/2024PASP..136a4202N
- El-Badry et al. (2024), *A population of neutron star candidates in wide orbits from Gaia astrometry*, OJAp 7, 58: https://ui.adsabs.harvard.edu/abs/2024OJAp....7E..58E
- Lam et al. (2025), *A Fast, Analytic Empirical Model of the Gaia Data Release 3 Astrometric Orbit Catalog Selection Function*, ApJ 987, 215: https://ui.adsabs.harvard.edu/abs/2025ApJ...987..215L
- Simon et al. (2026), *Radial Velocity Orbital Solutions for Candidate Black Hole and Neutron Star Binary Systems in the Gaia Data Release 3 Catalog*, ApJ 1007, 160: https://ui.adsabs.harvard.edu/abs/2026ApJ..1007..160S
- El-Badry et al. (2026), *Spectroscopic follow-up of compact object binary candidates from Gaia DR3: White dwarfs, neutron stars, black holes, and the parallax zeropoint*, arXiv:2608.06453 (preprint): https://ui.adsabs.harvard.edu/abs/2026arXiv260806453E
- El-Badry (2024), *Gaia's binary star renaissance*, New Astron. Rev. 98, 101694: https://ui.adsabs.harvard.edu/abs/2024NewAR..9801694E